<a href="https://colab.research.google.com/github/lizhuofan95/ASA2022_Workshop/blob/main/ASA_Working_001_20220706.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Introduction to Machine Learning into Qualitative Research

Coding allows us to see themes and patterns in qualitative data, but it can also get repetitive and dreary, especially with codes that are rather informational than interpretative. 

Previous approaches to automating qualitative coding rely heavily on prespecified rules, statistical assumptions, or dictionaries of keywords, often at the price of interpretative adaptability. 

Recent developments in deep learning models of natural languages provide a promising opportunity for qualitative researchers to scale their coding without compromising adaptability. Instead of following prespecified rules, deep learning algorithms are designed to mimic human behavior using human-generated examples. 

After iteratively establishing a codebook and some learning examples by hand-coding a sample of qualitative data, deep learning can help researchers quickly scale their initial codings on the remaining data and save up time and energy for thinking deeply about the data and the code. 

This Google Colab notebook walks you through the deep-learning-powered workflow we developed in our own study for analyzing in-depth interview data. 

## Introduction to Python in Google Colab

You are in Google Colab, or "Colaboratory", which allows you to write and execute Python in your browser without having to set up a programming environment on your own device. It also provides access to computational power for deep learning models free of charge for noncommercial uses. 

In this workshop, we use Google Colab and publicly available data to demonstrate the workflow, but you should NOT upload your own human subject data without a plan to protect their confidentiality! 

Now You can click the first sidebar button on the left to access the _table of Content_. Clicking "cell(s) hidden" will reveal more. You can also expand sections by clicking the small arrows to the left of headers. Try this now with the header "practice running code."  

### Running Python Code in Google Colab

You can execute embedded code by clicking on the arrow to the left of the cell. This is how we run programs or use functions. Try this below. 

In [ ]:
print ("Hello world, I am here to code your qualitative data more efficiently!")

Now run the simple program below. It will ask for your name, and say if it likes it.

In [ ]:
name = input('What is your name? ')
if name == 'Avery':
  print('Awesome name!', name, 'is pretty cool.')
elif name == "Zhuofan":
  print('Good name!', name, 'is pretty cool.')
else:
  print(' Well', name, ', your name is good I guess.')

What is your name? Zhuofan
Good name! Zhuofan is pretty cool.


Now that you understand the basics of this interface, the rest of this notebook will walk you through our deep-learning-powered workflow. 

## A Machine Learning Glossary

- Machine learning: the ability of an computer algorithm to build models based on sample data and make predictions and decisions on unseen data without being explicitly programmed to do so. An example is Google Search's autocomplete: When you type something in the search bar, it will predict the next word you are going to type and make suggestions. 

- Training data: a dataset of examples used by machine learning algorithms to learn patterns and build predictive models. 

- Test data: a dataset of examples used to provide an unbiased evaluation of how well trained models can make predictions. 

- Classification tasks: the task of assigning a class labels to input examples. An example is spam detection: many email services will classify incoming emails into spams and non-spams based on the content of the email. 

- Recall: measures how many percentage of all relevant cases a model picks up in a classification task. Suppose you are in a restaurant, a waiter misses three out of four items you ordered has a recall of 0.25, and of course you want a waiter with higher recall! 

- Precision: measures how many percentage of cases a model picks up are actually relevant in a classification task. Suppose you are in a restaurant, your waiter brings to you four items, but three out of the four items are items that you didn't order. This waiter has a precision of 25%, and of course you want a waiter with higher precision! 

- F-1: measures the overall performance of a machine learning model in classification tasks. 

## Overview of the Workflow

(1) Developing Codebook V.1

(2) Coding Training Data

(3) **Importing Data from ATLAS.ti into Python**

(4) **Preprocessing**

(5) **Scaling**

(6) **Recoding**

(7) **Reimporting Coded Data to ATLAS.ti**

(8) Examing Machine-Coded Data and Revising Codebook

(9) Repeat (2)-(8) with Codebook V.2...

Assuming that you have done an initial coding on a sample of the data, the following Python code does (3)-(7) for you. 

## Part I: Importing data from ATLAS.ti

Many qualitative researchers use QDA software to code their data, so here we start with an Excel spreadsheet that mimics the export function in ATLAS.ti. 

Assuming we code on the level of paragraphs, each row should contain all relevant information about a single paragraph. 

The columns should include:

- Interview ID
- Paragraph ID
- Paragraph Content
- Initial Codings

The rows should include:

- Training set: some paragraphs that you have "just coded"
- Testing set: some additional coded paragraphs that you set aside 
- Target set: the remaining uncoded paragraphs

Now let's look at a real example.

### Importing Data from ATLAS.ti

Assuming we have exported our interview data to an Excel file from ATLAS.ti or equivalent QDA software, we then tell the program where to find it. 

We can also specify the minimum length of each paragraph (in character), meaning any paragraph shorter than this minimum will be ignored. We set the number to be 50 characters, so that filler paragraphs such as "Yep", "Okay" and "I see" will not be used as training data. 

In [ ]:
path = "https://github.com/lizhuofan95/ASA2022_Workshop/blob/main/data/oralhistory_coded_short.xlsx?raw=true"

min_length = 50

This cell load the actual Excel file from our GitHub repository and executives the function above to import data into Python. 

In [ ]:
import pandas as pd  # The most commonly used library for data wrangling in Python

def read_ATLAS(path, min_length = -1):
    """
      Loads data ATLAS.ti-generated export files in .xls/.xlsx format, creates a copy that includes only participants' 
      speeches over a certain length for machine learning, and transforms the "Codes" column to one-hot encoding. 

      Args:
          path: file path to an ATLAS export file. It must have one paragraph per row and for each paragraph 
                include the following columns:

                    - Document: unique interview id
                    - Reference: unique pragraph id
                    - Quotation Content: paragraph content
                    - Codes: a list of codes that have been manually assigned to the paragraph

          min_length = the minimum number of characters that a paragraph must include to be included as valid data. This is to 
                       exclude filler sentences such as "yeah", "no", "I know" etc. Only comes into effect if you specify a 
                       value that is greater or equal to 1. 
      Returns: 
          original: the original, abbreviated version of the data, which we will use for reimporting machine-generated codings back into ATLAS.ti.
          training: the abbrevaited version of the data, which we will use for machine learning. 
          
    """
    data = pd.read_excel(path, engine='openpyxl', dtype = str)[['Document', 'Reference', 'Quotation Content', 'Codes']]
    
    data['Codes'] = data['Codes'].apply(lambda x: str(x).split("\n"))
    data['Quote'] = data['Quotation Content']

    data['Codes_Frozen'] = data['Codes'].apply(frozenset).to_frame(name='Codes_Frozen')
    for code in frozenset.union(*data.Codes_Frozen):
        data[code] = data.apply(lambda _: int(code in _.Codes_Frozen), axis=1)
    
    original = data

    # data['Quotation Content'] = data['Quotation Content'].apply(lambda x: str(x).split("\t"))
    # N = len(data)
    # data['Spk'] = ""
    # data['Quote'] = ""
    # for i in range(N):
    #     line = data['Quotation Content'][i]
    #     if len(line) >= 2:
    #         data['Spk'][i] = line[0].strip(":\s")
    #         data['Quote'][i] = line[1].strip('\u202c')
    #     elif len(line) == 1:
    #         data['Spk'][i] = ""
    #         data['Quote'][i] = line[0].strip()
    #     else:
    #         data['Spk'][i] = data['Quote'] = ""
    #     data['Codes'][i] = [data['Codes'][i][j].strip("#") for j in range(len(data['Codes'][i]))]
    # data = data[data["Spk"].isin(["MSPKR","FSPKR"])].reset_index(drop = True)
    # data = data[(data["Spk"] != "") & (data["INT_NEW"] != 1)].reset_index(drop = True)
    
    if min_length >= 0: 
        data = data[data['Quote'].map(len) >= min_length].reset_index(drop = True)
        
    training = data[[col for col in data.columns if col not in ['Quotation Content', 'Codes', 'Spk', 'Codes_Frozen', 'nan']]]
    
    return original, training

original, training = read_ATLAS(path = path, min_length = min_length)

NameError: ignored

This is how our data should look like in a machine-readable format. A wide range of computational tools will become available to you once you transform any text data into this format. 

In [ ]:
training

,Document,Reference,Quote,Background
0,Henry_B._Abajian,0,This is an interview with Henry Abajian on the...,1
1,Henry_B._Abajian,1,In 1938 I graduated with an electrical enginee...,1
2,Henry_B._Abajian,2,What were you particularly interested in at th...,1
3,Henry_B._Abajian,3,"It was all power engineering, and the electron...",0
4,Henry_B._Abajian,5,Yes. How I got there was interesting. The head...,0
...,...,...,...,...
19976,Anthony_Zimbalatti,177036,Before we talk about your Grumman experiences ...,0
19977,Anthony_Zimbalatti,177038,What at the time caused you and your colleague...,0
19978,Anthony_Zimbalatti,177039,I don't know. We always had these bull session...,0
19979,Anthony_Zimbalatti,177041,Both of us worked for Milt. I think he had an ...,0


## Preprocessing

For machine learning models to learn how humans code qualitative data, we also need to transform any raw text to something that computer can understand: numbers. 

People have come up with many ways of using numbers to represent text, but most of those systems are static and purely representational (e.g. Morse code), not for understanding the semantics and mimicing the human usage of human languages. 

Recent developments in natural language processing have nonetheless made many breakthroughs towards this direction, by using deep learning models trained on enourmous volumes of text data to generate high-dimensional vectors representing raw text such that the resulting representations can be used to learn and mimic how language is used. 

And our goal is precisely to mimic how human researchers code interviews. 

We use BERT, or Bidirectional Encoder Representation from Transformers, a family of deep learning language models trained on thousands of millions English words from sources like Google Books and Wikipedia to generate fine-grained and context-specific representations of language. 

### Set up a Deep Learning Environment

This cell sets up the computational environment in Google Colab. 

First, go to "Runtime - Change Runtime Type - Hardware Accelerator" and select "GPU". 

Then, make sure the `device` variable below equals to `"cuda"`. 

This will significantly speed up our deep learning model. We will also load necessary functionalities in this cell. 

In [ ]:
device = "cuda" # or "cpu"

!pip install transformers

import numpy as np

import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer

def text2feature(text, MAX_LENGTH = 512, BATCH_SIZE = 16, device = device):
    """
    Use pretrained BERT model to vectorize raw text. 
    
    Args:
        text: the text of interest, stored in the "Quote" column of our DataFrame "data_ML". 
        
        MAX_LENGTH: the maximum number of "wordpieces" (usually words but not always) in each paragraph to be used 
                    for representing the paragraph, capped at 512. 
        
        BATCH_SIZE: the maximum number of samples to be used in a single neural network iteration. It is recommended
                    to use a batch size of 32 or 64. We used 16 due to the limitation of our GPU memory. 
        
        device: "CPU" or "cuda". "cuda" is the architecture of the NVIDIA graphics processing unit (GPU) which is
                used to accelerate machine learning. Only use if you either (1) have a NVIDIA GPU and have installed 
                the CUDA development tools following the instruction or (2) use cloud computing (e.g. Google Colab). 
        
    Returns: 
        feature: an N_row by 768 array of vectors that represent each paragraph using a vector of 768.
    """

    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    
    encodings = list(map(lambda t: tokenizer.encode(t, padding=True, truncation=True, max_length = MAX_LENGTH, add_special_tokens=True), text))
    
    max_len = 0
    for i in encodings:
        if len(i) > max_len:
            max_len = len(i)

    encodings_padded = np.array([i + [0]*(max_len-len(i)) for i in encodings])
    attention_mask = [[float(i > 0) for i in ii] for ii in encodings_padded]
    
    dataset = TensorDataset(torch.tensor(encodings_padded, dtype = torch.int), torch.tensor(attention_mask))
    
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)
    
    model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)
    
    features = []

    with torch.no_grad():
        for step_num, batch_data in enumerate(dataloader):
            token_ids, masks = tuple(t.to(device) for t in batch_data)
            last_hidden_states = model(token_ids, masks)
            features.append(last_hidden_states[0][:,0,:].cpu().detach().numpy())
            """
            The model actually produces a vector of 768 for each of the 512 "wordpiece" in every sequence, but the 
            vector of every sequence, denoted by [CLS], is always a special classification token that can be used as
            the aggregate sequence representation for classification tasks. An alterantive is to represent the sequence
            by averaging all 512 vectors, which tends to produce similar results.           
            """
    features = np.vstack(features)
    
    return features

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


### From Text to Vectors Using Deep Learning

This cell executes the function above and transforms text data into vectors. We should obtain a N × 768 matrix that represents each row using a 768-dimensional vector. On average this will take about 5-7 minutes. 

In [ ]:
text = training['Quote'].values.tolist()
features = text2feature(text)
features.shape

Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/483 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/256M [00:00<?, ?B/s]

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_transform.bias', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_layer_norm.weight', 'vocab_transform.weight']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


(19981, 768)

Our data look like this in vectors:

In [ ]:
features

array([[-8.04037899e-02, -1.23658910e-01, -3.61755282e-01, ...,
        -4.42422088e-03,  5.73920608e-01,  4.51830387e-01],
       [ 1.52352005e-02,  7.93554708e-02, -3.65893543e-01, ...,
         2.96420068e-01,  3.53529871e-01,  6.38014317e-01],
       [ 1.67413235e-01, -4.30313461e-02, -2.37162262e-01, ...,
        -1.04779854e-01,  2.66423166e-01,  2.84095228e-01],
       ...,
       [-8.22876766e-02,  5.59788980e-02, -4.67753440e-01, ...,
         1.56504020e-01,  7.09437609e-01,  3.97252321e-01],
       [ 1.43082321e-01, -1.50504559e-01, -4.59693998e-01, ...,
        -5.96625090e-04,  4.93743062e-01,  5.25567591e-01],
       [-6.87612742e-02,  3.02690975e-02, -1.31134659e-01, ...,
        -1.04792535e-01,  1.61510170e-01,  2.77071506e-01]], dtype=float32)

### Generate Testing Samples for Evaluation

But before we extending our initial codings to all the remaining data, we want to have an sense of how reliable it would be. We do so by setting aside a small sample of coded data and compare machine-generated codings against our own codings on this already coded sample. And we want to do this multiple times to weed out the effect of outlier cases. 

We tell the algorithm how many times we want to test and how many percentage of the data we want to use for testing. 

The more data we use for testing, the less data we have left for training, the less accurate the scaling will be. 

In [ ]:
n_splits = 20
test_size = 0.75

In [ ]:
from sklearn.model_selection import ShuffleSplit

def split(all_cases, n_splits = 10, test_size = 0.25):
    """
    Split any coded data into training and test sets. 
    
    Args:
        n_splits: how many randomly reshuffled splits to generate. The default is 10. 
        
        test_size: how many cases to use as test data. 
                   - If between 0 and 1, represents the proportion of the dataset to include;
                   - If integer greater than 1, represents the the absolute number of test samples.
                   - The default is 25%. 
    Returns:
        train_set, test_set
    
    """
    train_test_split = ShuffleSplit(n_splits = n_splits, test_size = test_size)
    train_set = []
    test_set = []
    for train_index, test_index in train_test_split.split(all_cases):
        train_set.append(train_index.tolist())
        test_set.append(test_index.tolist())
    
    return train_set, test_set

all_cases = training['Document'].unique().tolist()

train_set, test_set = split(all_cases, n_splits, test_size)

## Scaling

Since BERT is typically deployed with data that are much bigger and deep learning models much more sophisticated than required by our tasks, here we simplify it by combining BERT vectors with a regularized logistic regression, which despite the strong assumption of linearity has been proven highly efficient for smaller scale data. 

First, we tell the algorithm which code we have already established on the training data and want to scale to the remaining data. 

In [ ]:
eval_codes = ['Background']

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, cohen_kappa_score

from typing import Union
from scipy.sparse import spmatrix

NDArray = Union[np.ndarray, spmatrix]

class Classifier:
    def __init__(self):
        
        self.clf = LogisticRegression(penalty = 'l2', solver = 'liblinear', C = 1, class_weight= 'balanced')
        
    def train(self, features: NDArray, labels: NDArray):

        self.clf.fit(features, labels)
    
    def predict(self, features: NDArray) -> NDArray:

        predictions = self.clf.predict(features)
        predprob = self.clf.predict_proba(features)
    
        return predictions, predprob
    
    def code(self, train_features, test_features, train_labels):
        
        self.train(train_features, train_labels)
        self.predicted_labels, self.predicted_probabilities = self.predict(test_features)
        
        return self.predicted_labels, self.predicted_probabilities
    
    def metrics(self, test_labels):
        
        accuracy = accuracy_score(test_labels, self.predicted_labels)
        f1 = f1_score(test_labels, self.predicted_labels, pos_label=1)
        precision = precision_score(test_labels, self.predicted_labels)
        recall = recall_score(test_labels, self.predicted_labels)
        kappa = cohen_kappa_score(test_labels, self.predicted_labels)
        
        return [accuracy, f1, precision, recall, kappa]

def classify(classifier, data, eval_codes, train_set, test_set, method = ['pred']):
    """
    Train a machine learning model to predict human codings of interest based on each training/test split.
    
    Args:
        classifier: the machine learning model. 
        
        eval_codes: the list of codes to be scaled, corresponding to the "Codes" column from the ATLAS.ti export file.
        
        train_set/test_set: the ids of interviews to use as training/test data. 
        
        method: "pred" or "eval". 
                
                - The "pred" method predicts codings on uncoded data without returning metrics. 
                - The "eval" method predicts codings on coded data and compare the predictions against original human codings. 
        
    Returns:
        predictions: the predicted probability of a code on a paragraph.
        
        metrics: a list of performance metrics the "eval" method generates. 
    
    """
    
    predictions = pd.DataFrame({'code': [], 'test_set': [], 'p': []})
    metrics = pd.DataFrame({'code': [], 'test_set': [], 'n':[], 'accuracy': [], 'f1': [], 'precision': [], 'recall': [],  
                            'kappa': []})
    
    for code in eval_codes:
        for train_ids, test_ids in zip(train_set, test_set):

            train_data = data[data['Document'].isin([all_cases[id] for id in train_ids])]
            test_data = data[-data['Document'].isin([all_cases[id] for id in train_ids])]

            train_id = train_data.index.tolist()
            test_id = test_data.index.tolist()

            train_features = features[train_id]
            test_features = features[test_id]

            train_labels = train_data[code].values.tolist()
            
            predicted_labels, predicted_probabilities = classifier.code(train_features, test_features, train_labels)

            predictions = predictions.append(pd.Series([code, test_ids] + [[x[1] for x in predicted_probabilities]], index = predictions.columns), ignore_index=True)

            if method == 'eval':
                test_labels = test_data[code].values.tolist()
                metrics = metrics.append(pd.Series([code, test_ids, data.loc[data['Document'].isin([all_cases[id] for id in train_ids]), code].sum()] + classifier.metrics(test_labels), index = metrics.columns), ignore_index=True) 
            
    return predictions, metrics

predictions, metrics = classify(Classifier(), training, eval_codes, train_set, test_set, 'eval')

pd.DataFrame({'N': training[eval_codes].sum()}).join(metrics.groupby('code')[['n', 'accuracy', 'f1', 'precision', 'recall', 'kappa']].mean())

,N,n,accuracy,f1,precision,recall,kappa
Background,3834,982.55,0.796048,0.566733,0.47664,0.701593,0.44001


## Recoding

Now that we have casted a wide net that presumbly catches about 70% of all paragraphs that would have been coded as "Educational Background", we want to quickly review the results and filter out obivous errors - especially the irrelevant ones that the algorithm has mistaken to be "Background".

This cell creates highly customized spreadsheets that include part of the original text that the algorithm has labeled as "Background". Researchers can then go through this list to correct any obvious _false positives_. 

In most cases, recoding machine predictions only takes a fraction of the time and effort required by full human coding, because the algorithm has filtered out the vast majority of "irrelevant" texts, as defined by _YOUR_ own coding. 

In [ ]:
from google.colab import files

test_set = metrics.sort_values(['recall', 'f1'], ascending = False).groupby('code').nth(0)['test_set'].tolist()[0]
train_set = list(set(range(len(all_cases)))-set(test_set))
predictions, _ = classify(Classifier(), training, eval_codes, [train_set], [test_set])

recode_set = test_set
combine_codes = [0]

sorting = [True]
sorting_vars = [['Background']]

pruning = [True]
pruning_thresholds = [0.5]

def gen_recode(data, predictions, all_cases, recode_set, combine_codes, sorting, pruning, sorting_vars = None, pruning_threshold = None):
    
    recode = data[data['Document'].isin([all_cases[id] for id in recode_set])][['Quote', 'Document', 'Reference']].reset_index(drop = True)
    
    for code in eval_codes:
        #recode_set = best.loc[code]['test_set']
        p = predictions.loc[(predictions['code'] == code) & (predictions['test_set'].apply(lambda x: x==recode_set))]['p'].values.tolist()[0]
        col = pd.DataFrame({str(combine_codes[eval_codes.index(code)]) + '_P_' + code :p})
        recode = recode.join(col)
    
    for doc_id in range(max(combine_codes)+1):
        out = recode[['Document', 'Reference']]
        name = "Pred_"

        rowids_to_keep = []

        for col in (col for col in recode if col.startswith(str(doc_id))):

            out = out.join(recode[[col]])

            colname = col.rsplit('_',1)[1]

            name = name + colname

            out.rename(columns={col: name}, inplace=True)

            out[colname+'?'] = ''

            if pruning[doc_id] == True:
                rowids_to_keep.append(out[name].index[out[name] >= pruning_threshold[doc_id]].tolist() )

        out = out.join(recode[['Quote']])

        if pruning[doc_id] == True:
            out = out.iloc[list(set([item for sublist in rowids_to_keep for item in sublist]))]
        
        if sorting[doc_id] == True:
            out = out.sort_values(by = ["Pred_" + v for v in sorting_vars[doc_id]], ascending = False)
        else:
            out = out.sort_index()

        print(out)
        #out.to_excel(name + ".xlsx")
        #files.download(name + ".xlsx")

gen_recode(training, predictions, all_cases, recode_set, combine_codes, sorting, pruning, sorting_vars, pruning_thresholds)

## Reimporting to ATLAS.ti

In [ ]:
def reimport2ATLAS(full_data, recoded, recoded_set, recoded_cases):
    for code in recoded_set:
        recoded.loc[recoded[code].isna(), code] = recoded.loc[recoded[code].isna(), "P_"+code]
        recoded[code] = recoded[code].apply(lambda x: x >= 0.5).astype(int)
        full_data.merge(recoded[['Document', 'Reference', code]], on = ['Document', 'Reference'], how = 'left')
   
    code_list = [col for col in full_data.columns.tolist() if col not in ['Document', 'Reference', 'Quote', 'Quotation Content', 'Codes', 'Codes_Frozen','nan']] 
    
    for i in range(len(full_data)):
        if full_data.loc[i,"Document"] in all_cases:
            for code in code_list:
                if(full_data.loc[i, code] >= 0.5): full_data.loc[i,'Quotation Content'] = full_data.loc[i,'Quotation Content'] + ' ' + '#' + code
        
    full_data['idx'] = full_data.groupby('Document').cumcount()
    full_data['p_idx'] = 'p' + full_data['idx'].astype(str)
    full_pivot = full_data.pivot(index='Document',columns='p_idx', values = 'Quotation Content')
    full_pivot = full_pivot.reindex(sorted(full_pivot.columns, key=lambda x: float(x[1:])), axis=1)
    full_pivot.to_excel("Recoded_output_transformed_all_cases_all_codes.xlsx")
    files.download("Recoded_output_transformed_all_cases_all_codes.xlsx")

# recoded_set = ["Background"]
# recoded = pd.read_excel("")
# reimport2ATLAS(original, recoded, recoded_set, all_cases)

## Appendix: Scraping Interview Data

### Install necessary libraries

In [ ]:
!pip install selenium
!apt-get update 
!apt install chromium-chromedriver

from selenium import webdriver
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

from bs4 import BeautifulSoup
import re

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 981 kB 34.1 MB/s 
     |████████████████████████████████| 358 kB 63.9 MB/s 
     |████████████████████████████████| 139 kB 75.9 MB/s 
     |████████████████████████████████| 55 kB 3.9 MB/s 
     |████████████████████████████████| 4.1 MB 66.0 MB/s 
     |████████████████████████████████| 58 kB 6.1 MB/s 
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
requests 2.23.0 requires urllib3!=1.25.0,!=1.25.1,<1.26,>=1.21.1, but you have urllib3 1.26.10 which is incompatible.
datascience 0.10.6 requires folium==0.2.1, but you have folium 0.8.3 which is incompatible.
Hit:1 https:/

### Get the Table of Content

In [ ]:
driver = webdriver.Chrome('chromedriver',chrome_options=chrome_options)
driver.get("https://ethw.org/Oral-History:List_of_all_Oral_Histories")
content = driver.page_source
soup = BeautifulSoup(content)

links = []

for link in soup.find_all('a', attrs={'href': re.compile("/Oral-History")}):
    links.append(link.get('href'))

links.index("/Oral-History:Henry_B._Abajian")

links[48:]

### Get Individual Oral Histories

In [ ]:
interviews = pd.DataFrame(columns = ['person', 'interview', 'bio', 'section', 'speaker', 'text'])

for link in links[48:50]:
    driver.get("https://ethw.org" + link)
    content = driver.page_source
    soup = BeautifulSoup(content)
    row = [[link.split(':')[1],
        soup.find('span', {'id': 'About_the_Interview'}).find_next('p').get_text() if soup.find('span', {'id': 'About_the_Interview'}) is not None else '', 
        soup.find_all('h2')[0].find_next('p').get_text() if soup.find_all('h2') is not None else '', 
        p.find_previous('h3').find_next('span', {'class' : 'mw-headline'}).get_text() if p.find_previous('h3') is not None else '',
        p.find('b').get_text(), 
        p.findNext('p').get_text()
    ] for p in soup.find_all('p') if (p.find('b') is not None) and (p.findNext('p') is not None)]
    interviews = pd.concat([interviews, pd.DataFrame(row, columns = ['person', 'interview', 'bio', 'section', 'speaker', 'text'])], axis = 0)

interviews = interviews.replace(r'\n',' ', regex=True)

interviews['speaker'] = interviews['speaker'].replace(r':', '', regex=True)

In [ ]:
interviews

NameError: ignored

In [ ]:
#interviews.to_excel("scraped_interviews.xlsx")
#files.download("scraped_interviews.xlsx")